# 🤖 Aula 05 — Estrutura de um Agente
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula05_ollama_langchain_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Construindo um agente de IA: de `requests` puro a LangChain, com memória e personalidade.

---


## 🏗️ Setup — Execute e prossiga

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

A célula abaixo faz tudo: instala, baixa o modelo e aquece.
Execute e avance — é boilerplate reutilizável.

In [ ]:
# ── Setup completo: Ollama + gemma4:e2b + warm up ────────────────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain-openai langchain-core requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelo
!ollama pull gemma4:e2b

# 6. Warm up (primeira inferência demora ~2-3 min)
print("🔥 Warm up...")
start = time.time()
!curl -s http://localhost:11434/api/chat \\
    -d '{"model":"gemma4:e2b","messages":[{"role":"user","content":"Oi"}],"stream":false,"keep_alive":-1}' \\
    > /dev/null
print(f"✅ Pronto em {time.time()-start:.1f}s")


## 0. Agente na Mão — `requests` puro

Antes de usar LangChain, vamos construir um agente **do zero** com o que já sabemos:
`requests.post()` + dicionários Python.

Assim fica claro o que o LangChain abstrai.

In [ ]:
import requests

def criar_agente(instrucoes, max_historico=20):
    return {
        "instrucoes": instrucoes,
        "historico": [],
        "max_historico": max_historico
    }

def conversar(agente, mensagem):
    # Monta a lista de mensagens
    messages = [{"role": "system", "content": agente["instrucoes"]}]

    # Pega só as últimas N mensagens do histórico
    limite = agente["max_historico"]
    for msg in agente["historico"][-limite:]:
        messages.append(msg)

    messages.append({"role": "user", "content": mensagem})

    # Chama o Ollama via endpoint OpenAI-compatible
    response = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={
            "model": "gemma4:e2b",
            "messages": messages,
            "stream": False
        }
    )

    resposta = response.json()["choices"][0]["message"]["content"]

    # Salva no histórico
    agente["historico"].append({"role": "user", "content": mensagem})
    agente["historico"].append({"role": "assistant", "content": resposta})

    return resposta

print("✅ Agente manual com requests pronto!")

In [ ]:
# Testando — o agente lembra do nome!
agente = criar_agente("Você é um assistente amigável que responde em português.")

print(conversar(agente, "Oi! Meu nome é Ana."))
print()
print(conversar(agente, "Qual é o meu nome?"))

## 1. LangChain + OpenAI-compatible

Montar mensagens na mão funciona, mas é repetitivo. **LangChain** abstrai esse trabalho.

O Ollama fornece um endpoint compatível com a API da OpenAI em `/v1/chat/completions`. Com `ChatOpenAI` do `langchain-openai`, a gente usa esse endpoint **direto** — mesmo modelo local, sem enviar nada pra nuvem.

> **Por que `ChatOpenAI` e não `ChatOllama`?**
> O `ChatOllama` existe, mas o endpoint OpenAI-compatible é mais universal: funciona com Ollama, com qualquer servidor vLLM/TGI, e com APIs comerciais (Gemini, OpenAI). É o padrão da indústria — aprender assim te prepara pra trocar de provedor mudando uma linha.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gemma4:e2b",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",  # Ollama local não usa chave
    temperature=0.7
)

resposta = llm.invoke("O que é LangChain em uma frase?")
print(f"🤖 {resposta.content}")


## 3. Chat com memória — LangChain

Na seção 0, gerenciamos o histórico manualmente com listas de dicts.
No LangChain, podemos fazer o mesmo usando `InMemoryChatMessageHistory` —
que é basicamente uma lista tipada. A lógica é **idêntica**:

1. Guardar mensagens no histórico
2. Passar o histórico para o modelo
3. Salvar a resposta de volta no histórico

⚠️ `RunnableWithMessageHistory` (a abordagem anterior) foi **deprecated** no
LangChain v0.3. A abordagem recomendada é gerenciar o histórico explicitamente, como
fazemos abaixo — mais claro e mais fácil de debugar.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

# Sessão com histórico (sem gerenciamento de sessões — uma variável simples)
session = InMemoryChatMessageHistory()

# Prompt com placeholder para o histórico
prompt_mem = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente amigável que responde em português."),
    ("placeholder", "{history}"),
    ("human", "{pergunta}")
])

cadeia_mem = prompt_mem | llm

print("✅ Cadeia com memória pronta!")

In [ ]:
# Primeira mensagem — adicionamos ao histórico manualmente
r1 = cadeia_mem.invoke({
    "pergunta": "Meu nome é Ana e eu estudo engenharia.",
    "history": session.messages  # histórico vazio na primeira vez
})
print(f"🤖 {r1.content}")

# Salva no histórico
session.add_message(HumanMessage(content="Meu nome é Ana e eu estudo engenharia."))
session.add_message(AIMessage(content=r1.content))

In [ ]:
# Segunda mensagem — o modelo lembra do nome!
r2 = cadeia_mem.invoke({
    "pergunta": "Qual é o meu nome e o que eu estudo?",
    "history": session.messages  # agora o histórico tem as mensagens anteriores
})
print(f"🤖 {r2.content}")

# Salva no histórico
session.add_message(HumanMessage(content="Qual é o meu nome e o que eu estudo?"))
session.add_message(AIMessage(content=r2.content))

In [ ]:
# Sessão zerada = sem memória
session_nova = InMemoryChatMessageHistory()

r3 = cadeia_mem.invoke({
    "pergunta": "Qual é o meu nome?",
    "history": session_nova.messages  # histórico vazio!
})
print(f"🤖 {r3.content}")
print("👉 Na sessão nova o modelo não sabe nosso nome!")

---
✅ Resumo do que vimos:
- **requests puro** → agente com dict + funções (sem abstração)
- **ChatOpenAI** → conecta LangChain ao LLM via endpoint OpenAI-compatible
- **ChatPromptTemplate** → prompts reutilizáveis com variáveis
- **InMemoryChatMessageHistory** → histórico tipado (mesma lógica da seção 0)
- O padrão `prompt | llm` é a base de tudo no LangChain

Na S06 vamos adicionar **ferramentas** (calculadora, busca) ao agente!

*Material da Guilda de IA — UFVJM 2026.1*